# ML Practical 2 — Email Spam Classification using KNN and SVM

**Objective:** Classify emails as Spam (1) or Not Spam (0) using binary classification.  
**Algorithms:** K-Nearest Neighbors (KNN) and Support Vector Machine (SVM)  
**Dataset:** emails.csv — place the file downloaded from Kaggle in this folder.  
**Source:** https://www.kaggle.com/datasets/balaka18/email-spam-classification-dataset-csv


---
## Step 1 — Import Libraries

We import:
- `pandas` and `numpy` for data loading and manipulation
- `train_test_split` to split data into training and test sets
- `StandardScaler` to normalize the feature values
- `KNeighborsClassifier` for the KNN model
- `SVC` (Support Vector Classifier) for the SVM model
- `confusion_matrix`, `accuracy_score`, `classification_report` for evaluating both models
- `matplotlib` and `seaborn` for visualization

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    classification_report, ConfusionMatrixDisplay
)

print('Libraries imported successfully')

---
## Step 2 — Load the Dataset

We read the CSV file into a DataFrame.  
`shape` tells us the number of rows (emails) and columns (word features + label).  
`head()` shows a preview of the first 5 rows.

In [ ]:
# Place emails.csv (downloaded from Kaggle) in this folder
df = pd.read_csv('emails.csv')

print('Shape:', df.shape)
df.head()

---
## Step 3 — Explore the Dataset

`info()` shows the data types and confirms there are no null values.  
`value_counts()` on the `spam` column shows how many emails are spam vs not spam.  
A bar chart visualizes this class distribution.

In [ ]:
df.info()

In [ ]:
print('Missing values:\n', df.isnull().sum().sum())
print('\nClass distribution:')
print(df['spam'].value_counts())

In [ ]:
df['spam'].value_counts().plot(kind='bar', color=['steelblue', 'salmon'])
plt.title('Email Class Distribution')
plt.xlabel('spam (0 = Not Spam, 1 = Spam)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

---
## Step 4 — Separate Features and Target

The dataset contains word-frequency columns as features.  
- We drop the `Email No.` column — it is just an identifier with no predictive value.
- All remaining columns except `spam` become the feature matrix **X**.
- The `spam` column becomes the target vector **y** (0 = Not Spam, 1 = Spam).

In [ ]:
# Drop identifier column
df.drop(columns=['Email No.'], inplace=True)

X = df.drop(columns=['spam'])
y = df['spam']

print('Features shape:', X.shape)
print('Target shape  :', y.shape)

---
## Step 5 — Split into Training and Test Sets

We split 80% of data for training and 20% for testing.  
`stratify=y` ensures both sets have the same spam ratio as the full dataset.  
`random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training samples:', X_train.shape[0])
print('Testing  samples:', X_test.shape[0])

---
## Step 6 — Normalize the Data

Both KNN and SVM are distance-based algorithms — they are sensitive to the scale of features.  
`StandardScaler` scales all features to have mean=0 and standard deviation=1.  
We fit the scaler only on training data to avoid leaking information from the test set.

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print('Data normalized successfully')

---
## Step 7 — Model 1: K-Nearest Neighbors (KNN)

KNN classifies a point by looking at the **K closest training examples** and taking a majority vote.  
- `n_neighbors=5` means we look at the 5 nearest neighbors to decide the class.
- `metric='minkowski'` with `p=2` is equivalent to Euclidean distance.

We train the model on the training set and predict on the test set.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)

knn_acc = accuracy_score(y_test, y_pred_knn)
print(f'KNN Accuracy: {knn_acc * 100:.2f}%')

---
## Step 8 — Find the Best K for KNN

Instead of just using K=5, we test K values from 1 to 20 and plot the accuracy for each.  
This helps us pick the optimal K that gives the best test accuracy without overfitting.

In [ ]:
k_values   = range(1, 21)
accuracies = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    accuracies.append(accuracy_score(y_test, model.predict(X_test)))

best_k   = k_values[accuracies.index(max(accuracies))]
best_acc = max(accuracies)
print(f'Best K: {best_k}  |  Accuracy: {best_acc * 100:.2f}%')

plt.figure(figsize=(10, 4))
plt.plot(k_values, accuracies, marker='o', color='steelblue')
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.title('KNN Accuracy vs Number of Neighbors (K)')
plt.xlabel('K')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

---
## Step 9 — KNN Confusion Matrix and Classification Report

We retrain KNN with the best K and evaluate it fully.  
The confusion matrix shows exactly where the model is right and wrong.  
The classification report gives precision, recall, and F1-score for each class.

In [ ]:
# Retrain with best K
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train, y_train)
y_pred_knn = knn_best.predict(X_test)

# Confusion Matrix
cm_knn = confusion_matrix(y_test, y_pred_knn)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Spam', 'Spam'],
            yticklabels=['Not Spam', 'Spam'])
plt.title(f'KNN Confusion Matrix (K={best_k})')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

print(f'KNN Accuracy : {accuracy_score(y_test, y_pred_knn) * 100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_knn, target_names=['Not Spam', 'Spam']))

---
## Step 10 — Model 2: Support Vector Machine (SVM)

SVM finds the **optimal hyperplane** that best separates the two classes with the maximum margin.  
- `kernel='rbf'` (Radial Basis Function) maps the data to a higher-dimensional space where it may be linearly separable — useful for non-linear data like text features.
- `C=1` is the regularization parameter — higher C = less margin but fewer misclassifications.
- `gamma='scale'` automatically sets gamma based on the number of features.

In [ ]:
svm = SVC(kernel='rbf', C=1, gamma='scale', random_state=42)
svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

svm_acc = accuracy_score(y_test, y_pred_svm)
print(f'SVM Accuracy: {svm_acc * 100:.2f}%')

---
## Step 11 — SVM Confusion Matrix and Classification Report

Same evaluation as KNN — we look at the confusion matrix and the full classification report.

In [ ]:
cm_svm = confusion_matrix(y_test, y_pred_svm)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Not Spam', 'Spam'],
            yticklabels=['Not Spam', 'Spam'])
plt.title('SVM Confusion Matrix (RBF Kernel)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

print(f'SVM Accuracy : {svm_acc * 100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_svm, target_names=['Not Spam', 'Spam']))

---
## Step 12 — Compare KNN vs SVM

We compare both models side by side using accuracy.  
A bar chart makes the comparison visual and easy to interpret.

In [ ]:
models      = ['KNN (Best K)', 'SVM (RBF)']
accuracies_final = [
    accuracy_score(y_test, y_pred_knn) * 100,
    accuracy_score(y_test, y_pred_svm) * 100
]

plt.figure(figsize=(6, 4))
bars = plt.bar(models, accuracies_final, color=['steelblue', 'darkorange'], width=0.4)
for bar, acc in zip(bars, accuracies_final):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 2,
             f'{acc:.2f}%', ha='center', va='top', color='white', fontweight='bold')
plt.ylim(80, 100)
plt.title('KNN vs SVM — Accuracy Comparison')
plt.ylabel('Accuracy (%)')
plt.show()

print(f'KNN Accuracy : {accuracies_final[0]:.2f}%')
print(f'SVM Accuracy : {accuracies_final[1]:.2f}%')
winner = models[accuracies_final.index(max(accuracies_final))]
print(f'\nBetter Model : {winner}')

---
## Summary

| Step | What we did |
|------|-------------|
| 1 | Imported libraries |
| 2 | Loaded emails.csv |
| 3 | Explored class distribution |
| 4 | Separated features (X) and target (y), dropped Email No. |
| 5 | Split into 80% train / 20% test with stratification |
| 6 | Normalized using StandardScaler |
| 7–9 | Trained KNN, found best K, evaluated with confusion matrix |
| 10–11 | Trained SVM with RBF kernel, evaluated with confusion matrix |
| 12 | Compared both models side by side |

**Key differences between KNN and SVM:**
- KNN is a **lazy learner** — stores all training data and classifies by proximity at prediction time. Simple but slow for large datasets.
- SVM is an **eager learner** — builds a model (hyperplane) during training. Faster at prediction and generally more robust on high-dimensional data like word features.
